In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.feature_selection import RFE
from xgboost import XGBClassifier
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset    

In [ ]:
RANDOM_STATE = 42
MAX_ITER = 1000
N_ESTIMATORS = 100
torch.manual_seed(42)
data_path = "C:/GitHub/telecom_churn_predictor/data/processed_data.csv"
df = pd.read_csv(data_path)

In [ ]:
X = df.drop("churn_probability", axis=1)
y = df["churn_probability"]

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=RANDOM_STATE, stratify=y_temp)
print(f"Split done: Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
print(f"Before resampling: {y_train.value_counts()}")

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

smote = SMOTE(random_state=RANDOM_STATE)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)
print(f"After resampling: {pd.Series(y_train_resampled).value_counts()}")

In [ ]:
lr = LogisticRegression(max_iter=MAX_ITER, random_state=RANDOM_STATE)
lr.fit(X_train_resampled, y_train_resampled)

y_val_pred = lr.predict(X_val_scaled)
y_val_proba = lr.predict_proba(X_val_scaled)[:, 1]
print("Validation Classification Report:")
print(classification_report(y_val, y_val_pred))
print("Validation ROC AUC:", roc_auc_score(y_val, y_val_proba))

In [ ]:
rf = RandomForestClassifier(n_estimators=N_ESTIMATORS, random_state=RANDOM_STATE)
rf.fit(X_train_resampled, y_train_resampled)

y_val_pred_rf = rf.predict(X_val_scaled)
y_val_proba_rf = rf.predict_proba(X_val_scaled)[:, 1]
print("Validation Classification Report (Random Forest):")
print(classification_report(y_val, y_val_pred_rf))
print("Validation ROC AUC (Random Forest):", roc_auc_score(y_val, y_val_proba_rf))

In [ ]:
xgb = XGBClassifier(n_estimators=N_ESTIMATORS, random_state=RANDOM_STATE, eval_metric='logloss')
xgb.fit(X_train_resampled, y_train_resampled)

y_val_pred_xgb = xgb.predict(X_val_scaled)
y_val_proba_xgb = xgb.predict_proba(X_val_scaled)[:, 1]
print("Validation Classification Report (XGBoost):")
print(classification_report(y_val, y_val_pred_xgb))
print("Validation ROC AUC (XGBoost):", roc_auc_score(y_val, y_val_proba_xgb))

In [ ]:
from models import ChurnPredictorNN
nn_model = ChurnPredictorNN(input_dim=X_train_resampled.shape[1], hidden_layers=[128, 64, 32],
                             output_dim=1)

print(nn_model)

In [ ]:
X_train_tensor = torch.tensor(X_train_resampled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_resampled.values, dtype=torch.float32).unsqueeze(1)
X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32).unsqueeze(1)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

loss = nn.BCELoss()
optimizer = torch.optim.Adam(nn_model.parameters(), lr=0.001)

In [ ]:
EPOCHS = 30
for epoch in range(EPOCHS):
    nn_model.train()
    epoch_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = nn_model(X_batch)
        batch_loss = loss(outputs, y_batch)
        batch_loss.backward()
        optimizer.step()
        epoch_loss += batch_loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {epoch_loss/len(train_loader):.4f}")


In [ ]:
nn_model.eval()
with torch.no_grad():
    y_val_proba_nn = nn_model(X_val_tensor).squeeze().numpy()
    y_val_pred_nn = (y_val_proba_nn >= 0.5).astype(int)

print("Validation Classification Report (Neural Network):")
print(classification_report(y_val, y_val_pred_nn))
print("Validation ROC AUC (Neural Network):", roc_auc_score(y_val, y_val_proba_nn))

In [ ]:
y_test_pred_rf = rf.predict(X_test_scaled)
y_test_proba_rf = rf.predict_proba(X_test_scaled)[:, 1]

print("Test Classification Report (Random Forest):")
print(classification_report(y_test, y_test_pred_rf))
print("Test ROC AUC (Random Forest):", roc_auc_score(y_test, y_test_proba_rf))

In [ ]:
feature_names = X.columns.tolist()
importances = rf.feature_importances_

feature_df = pd.DataFrame({"feature": feature_names, "importance": importances})
feature_df = feature_df.sort_values(by="importance", ascending=False)
print(feature_df.head(10))
print(feature_df[feature_df['feature'].str.contains('trend')])



In [ ]:
rfe = RFE(estimator=RandomForestClassifier(n_estimators=50, random_state=RANDOM_STATE), 
          n_features_to_select=20)
rfe.fit(X_train_resampled, y_train_resampled)

X_val_rfe = rfe.transform(X_val_scaled)
X_test_rfe = rfe.transform(X_test_scaled)

rf_rfe = RandomForestClassifier(n_estimators=N_ESTIMATORS, random_state=RANDOM_STATE)
rf_rfe.fit(rfe.transform(X_train_resampled), y_train_resampled)

y_test_pred_rfe = rf_rfe.predict(X_test_rfe)
y_test_proba_rfe = rf_rfe.predict_proba(X_test_rfe)[:, 1]
print("Test Classification Report (Random Forest with RFE):")
print(classification_report(y_test, y_test_pred_rfe))
print("Test ROC AUC (Random Forest with RFE):", roc_auc_score(y_test, y_test_proba_rfe))

In [ ]:
selected_features = X.columns[rfe.support_].tolist()
print("Selected features after RFE:", selected_features)

results_df = X_test[selected_features].copy()
results_df['actual_churn'] = y_test.values
results_df['predicted_churn'] = y_test_pred_rfe
results_df['churn_probability'] = y_test_proba_rfe

results_df.to_csv('data/churn_predictions.csv', index=False)
print("Predictions saved to data/churn_predictions.csv")

In [ ]:
feat_importance_df = pd.DataFrame({
    'feature': selected_features,
    'importance': rf_rfe.feature_importances_
}).sort_values('importance', ascending=False)

feat_importance_df.to_csv('data/feature_importance.csv', index=False)
print(feat_importance_df)